# Photo Recall Discovery Engine: data collection

Run the cells top to bottom (Runtime > Run all). The last cell downloads `corpus.csv`.
Upload that file in the app's **Run the pipeline** tab.

Sources: Google Play reviews, App Store reviews, Reddit posts and comments, plus any threads you collect by hand (Google Photos Community, YouTube comments).

In [ ]:
!pip install -q google-play-scraper
import re, time, json, requests
import pandas as pd
from google_play_scraper import reviews, Sort

## Settings

In [ ]:
PLAY_APP_ID = "com.google.android.apps.photos"
APPSTORE_APP_ID = "962194608"          # Google Photos on the App Store
PLAY_COUNTRIES = ["in", "us"]
APPSTORE_COUNTRIES = ["in", "us", "gb"]
PLAY_REVIEWS_PER_COUNTRY = 20000       # newest reviews scanned per country

SUBREDDITS = ["googlephotos", "GooglePixel", "Android"]
REDDIT_QUERIES = [
    "can't find photo", "find old photo", "search not working",
    "ask photos", "looking for a photo", "search screenshot",
    "find picture", "photo search",
    "ask photos vs classic", "ask photos not working", "turn off ask photos",
    "classic search toggle", "ai search worse", "gemini search photos",
]
COMMENTS_PER_POST = 15

# App reviews are only kept if they mention finding or searching.
FIND_WORDS = re.compile(
    r"(can'?t find|cannot find|couldn'?t find|unable to find|not able to find|hard to find|"
    r"find (a|an|the|my|old|that|specific|any)\b|search|looking for|remember|scroll|"
    r"lost (a |my |all )?(photo|pic|picture|video)|ask photos|classic search|gemini|"
    r"where (is|are) my|"
    r"dhund|nahi mil)",   # common Hinglish: "dhundh", "nahi mil raha"
    re.I,
)
def mentions_finding(text):
    return bool(FIND_WORDS.search(str(text).replace("\u2019", "'")))

## 1. Google Play reviews

In [ ]:
play_rows = []
for country in PLAY_COUNTRIES:
    for sort in (Sort.NEWEST, Sort.MOST_RELEVANT):
        result, _ = reviews(PLAY_APP_ID, lang="en", country=country, sort=sort,
                            count=PLAY_REVIEWS_PER_COUNTRY if sort == Sort.NEWEST else 3000)
        for r in result:
            play_rows.append({
                "source": "play_store", "native_id": r["reviewId"], "text": r["content"],
                "rating": r["score"], "date": str(r["at"])[:10], "url": "",
                "country": country,
            })
        print(country, sort, len(result))
play = pd.DataFrame(play_rows).drop_duplicates("native_id")
play = play[play["text"].apply(mentions_finding)]
print("Play Store reviews kept:", len(play))

## 2. App Store reviews

In [ ]:
ios_rows = []
for country in APPSTORE_COUNTRIES:
    for page in range(1, 11):
        url = (f"https://itunes.apple.com/{country}/rss/customerreviews/"
               f"page={page}/id={APPSTORE_APP_ID}/sortby=mostrecent/json")
        try:
            entries = requests.get(url, timeout=20).json().get("feed", {}).get("entry", [])
        except Exception as e:
            print("skip", country, page, e); break
        if isinstance(entries, dict):
            entries = [entries]
        if not entries:
            break
        for e in entries:
            if "content" not in e:
                continue
            ios_rows.append({
                "source": "app_store", "native_id": e["id"]["label"],
                "text": f"{e['title']['label']}. {e['content']['label']}",
                "rating": int(e["im:rating"]["label"]), "date": e["updated"]["label"][:10],
                "url": "", "country": country,
            })
        time.sleep(1)
ios = pd.DataFrame(ios_rows).drop_duplicates("native_id")
ios = ios[ios["text"].apply(mentions_finding)] if len(ios) else ios
print("App Store reviews kept:", len(ios))

## 3. Reddit posts and comments

Uses Reddit's public JSON. If you see lots of `403` or `429` errors, Reddit is blocking Colab: run the optional PRAW cell below instead.

In [ ]:
HEADERS = {"User-Agent": "photo-recall-research/1.0 (student research project)"}

def get_json(url, tries=3):
    for i in range(tries):
        r = requests.get(url, headers=HEADERS, timeout=20)
        if r.status_code == 200:
            return r.json()
        print("  status", r.status_code, "retrying...")
        time.sleep(5 * (i + 1))
    return None

posts = {}
for sub in SUBREDDITS:
    for q in REDDIT_QUERIES:
        url = (f"https://www.reddit.com/r/{sub}/search.json?q={requests.utils.quote(q)}"
               f"&restrict_sr=1&sort=relevance&t=all&limit=100")
        data = get_json(url)
        if not data:
            continue
        for child in data["data"]["children"]:
            p = child["data"]
            posts[p["id"]] = p
        time.sleep(2)
print("Reddit posts found:", len(posts))

reddit_rows = []
for i, (pid, p) in enumerate(posts.items()):
    link = "https://www.reddit.com" + p["permalink"]
    reddit_rows.append({
        "source": "reddit_post", "native_id": pid,
        "text": f"{p['title']}. {p.get('selftext', '')}",
        "rating": None, "date": pd.to_datetime(p["created_utc"], unit="s").strftime("%Y-%m-%d"),
        "url": link, "country": "",
    })
    thread = get_json(link.rstrip("/") + ".json?limit=50&depth=1")
    if thread and len(thread) > 1:
        kept = 0
        for c in thread[1]["data"]["children"]:
            body = c.get("data", {}).get("body", "")
            if c.get("kind") != "t1" or len(body) < 60 or body in ("[deleted]", "[removed]"):
                continue
            reddit_rows.append({
                "source": "reddit_comment", "native_id": c["data"]["id"],
                "text": f"(Reply to: {p['title']}) {body}",
                "rating": None,
                "date": pd.to_datetime(c["data"]["created_utc"], unit="s").strftime("%Y-%m-%d"),
                "url": "https://www.reddit.com" + c["data"]["permalink"], "country": "",
            })
            kept += 1
            if kept >= COMMENTS_PER_POST:
                break
    time.sleep(2)
    if i % 25 == 0:
        print(f"  threads read: {i}/{len(posts)}")
reddit = pd.DataFrame(reddit_rows)
print("Reddit rows:", len(reddit))

### Optional: Reddit via PRAW (only if the cell above was blocked)

Create a free "script" app at https://www.reddit.com/prefs/apps and paste the client id and secret below.

In [ ]:
USE_PRAW = False
if USE_PRAW:
    !pip install -q praw
    import praw
    reddit_api = praw.Reddit(client_id="YOUR_ID", client_secret="YOUR_SECRET",
                             user_agent="photo-recall-research/1.0")
    reddit_rows = []
    for sub in SUBREDDITS:
        for q in REDDIT_QUERIES:
            for s in reddit_api.subreddit(sub).search(q, sort="relevance", time_filter="all", limit=100):
                reddit_rows.append({"source": "reddit_post", "native_id": s.id,
                                    "text": f"{s.title}. {s.selftext}", "rating": None,
                                    "date": pd.to_datetime(s.created_utc, unit="s").strftime("%Y-%m-%d"),
                                    "url": "https://www.reddit.com" + s.permalink, "country": ""})
                s.comments.replace_more(limit=0)
                for c in s.comments[:COMMENTS_PER_POST]:
                    if len(c.body) >= 60:
                        reddit_rows.append({"source": "reddit_comment", "native_id": c.id,
                                            "text": f"(Reply to: {s.title}) {c.body}", "rating": None,
                                            "date": pd.to_datetime(c.created_utc, unit="s").strftime("%Y-%m-%d"),
                                            "url": "https://www.reddit.com" + c.permalink, "country": ""})
    reddit = pd.DataFrame(reddit_rows).drop_duplicates("native_id")
    print("Reddit rows:", len(reddit))

## 4. Hand-collected threads (optional)

Google Photos Community threads and YouTube comments are hard to scrape reliably. Copy the useful ones into a Google Sheet with columns `source, text, url, date`, download it as CSV, and upload it here. Use sources like `gp_community` or `youtube`.

In [ ]:
manual = pd.DataFrame()
try:
    from google.colab import files
    print("Upload a manual CSV, or click Cancel to skip.")
    up = files.upload()
    if up:
        name = list(up)[0]
        manual = pd.read_csv(name)
        manual["native_id"] = [f"m{i}" for i in range(len(manual))]
        print("Manual rows:", len(manual))
except Exception as e:
    print("Skipped manual upload:", e)

## 5. Combine and download

In [ ]:
PREFIX = {"play_store": "ps", "app_store": "as", "reddit_post": "rp", "reddit_comment": "rc"}
frames = [f for f in (play, ios, reddit, manual) if len(f)]
corpus = pd.concat(frames, ignore_index=True)
corpus["text"] = corpus["text"].astype(str).str.strip()
corpus = corpus[corpus["text"].str.len() >= 30]
corpus = corpus.drop_duplicates("text").reset_index(drop=True)
corpus["id"] = [f"{PREFIX.get(s, 'mn')}-{i:05d}" for i, s in enumerate(corpus["source"])]
corpus = corpus[["id", "source", "text", "url", "date", "rating", "country"]]
print(corpus["source"].value_counts())
print("Total:", len(corpus))
corpus.to_csv("corpus.csv", index=False)
try:
    from google.colab import files
    files.download("corpus.csv")
except Exception:
    print("Saved corpus.csv")